# 🌊 astream_events 为什么输出这么长？

> `astream_events` 把 LangGraph / LangChain 运行过程中 **所有底层事件** 都 stream 出来。

## 📋 事件类型一览

你贴出来的 output 里主要有这几类事件：

```text
on_chain_start
on_chat_model_start
on_chat_model_stream   ← 这个最多！
on_chat_model_end
on_chain_end
```

---

## 🔑 为什么 `on_chat_model_stream` 出现这么多次？

因为它是 **模型 token streaming 事件**。

模型不是一次性吐出完整回答，而是：

```text
chunk 1: "The"
chunk 2: " San"
chunk 3: " Francisco"
chunk 4: " 49ers"
chunk 5: " are"
...
```

每生成一点点，就触发一次 `on_chat_model_stream`。

如果回答比较长，模型可能生成几百个小 chunk → 几百行输出。

**这就是 output 爆长的原因。** 不是因为 graph 循环了！

---

## 📊 四种 stream 方式对比

| 方式 | 含义 |
|------|------|
| `graph.stream(..., stream_mode="updates")` | 每个 node 返回了什么 |
| `graph.stream(..., stream_mode="values")` | 每一步之后完整 state 是什么 |
| `graph.astream_events(...)` | node 开始、模型开始、模型每个 token、模型结束、node 结束、graph 结束 |

```text
astream_events = 全量运行日志（最详细）
updates = delta（每个 node 的输出）
values = 完整 state 快照
```

---

## 🔍 一次运行的事件流

```text
LangGraph start
↓
conversation node start
↓
ChatOpenAI start
↓
ChatOpenAI stream chunk 1
ChatOpenAI stream chunk 2
ChatOpenAI stream chunk 3
...（几百个 chunks）
↓
ChatOpenAI end
↓
should_continue start → end
↓
conversation node end
↓
LangGraph end
```

---

## ✅ 解决方案

### 方案 1：只看最终结果，用 `astream` + `updates`

In [ ]:
# 如果只想看最终 AIMessage，不要用 astream_events
# async for chunk in graph.astream(
#     {"messages": [input_message]},
#     config,
#     stream_mode="updates",
# ):
#     print(chunk)

### 方案 2：用 `astream_events` 但过滤掉 token chunks

In [ ]:
# async for event in graph.astream_events(
#     {"messages": [input_message]},
#     config,
#     version="v2",
# ):
#     if event["event"] == "on_chat_model_stream":
#         continue  # 跳过 token chunks
#
#     print(
#         f"Node: {event['metadata'].get('langgraph_node', '')}. "
#         f"Type: {event['event']}. "
#         f"Name: {event['name']}"
#     )

这样输出会短很多。✅

---

## 💡 总结

> 输出长不是因为 graph 循环了，而是因为 **一个 ChatOpenAI 调用内部产生了很多 stream chunk events**。

选择合适的 stream 方式，按需获取信息。🎯